# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nimrafarooq2006-coder/ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row means: one web page (content item) for March 2026. The
warehouse table has one row per day, per client, per page, so I add
the days up into one row per page.

Table I use: fact_content_daily_performance, for month=2026-03.

Time window: features come from March 1 to 14 (the decision moment is
March 14). The outcome is measured from March 15 to 31.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

What I rank: pages that may be losing visibility. The proxy label is
"declining": average daily impressions in March 15-31 are lower than
in March 1-14, for pages with enough impressions to matter.Only pages with at least 100 impressions in March 1-14 are included.

What I exclude: anything from March 15-31 as a feature (it is the
future), and the final month (June 2026), which I keep sealed as a
test month.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
path = f"{rel}/fact_content_daily_performance/month=2026-03/*.parquet"


In [10]:
base = f"read_parquet('{path}')"

print("QUERY 1: the grain")
con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS distinct_day_client_page
FROM {base}
""").show()

print("QUERY 2: row count and date span")
con.sql(f"""
SELECT COUNT(*) AS total_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
       COUNT(DISTINCT content_hash_id) AS pages, COUNT(DISTINCT client_hash_id) AS clients
FROM {base}
""").show()

print("QUERY 3: availability, with IS TRUE")
con.sql(f"""
SELECT (SELECT COUNT(*) FROM {base}) AS total_rows,
       COUNT(*) AS rows_surviving
FROM {base}
WHERE gsc_data_available IS TRUE
""").show()

QUERY 1: the grain


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────────┐
│ total_rows │ distinct_day_client_page │
│   int64    │          int64           │
├────────────┼──────────────────────────┤
│    9841378 │                  9841378 │
└────────────┴──────────────────────────┘

QUERY 2: row count and date span
┌────────────┬────────────┬────────────┬────────┬─────────┐
│ total_rows │ first_day  │  last_day  │ pages  │ clients │
│   int64    │    date    │    date    │ int64  │  int64  │
├────────────┼────────────┼────────────┼────────┼─────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │ 331437 │      55 │
└────────────┴────────────┴────────────┴────────┴─────────┘

QUERY 3: availability, with IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ rows_surviving │
│   int64    │     int64      │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘



Query 1 (grain): total rows and distinct day + client + page are the
same (9,841,378), so one row really is one day for one client and one
page.

Query 2 (size and dates): 9,841,378 rows from 2026-03-01 to
2026-03-31, covering 331,437 pages and 55 clients.

Query 3 (availability): filtering with IS TRUE keeps 3,611,061 of
9,841,378 rows (about 37%). The rest have no search data, so I leave
them out.

In [11]:
raw = con.sql(f"""
SELECT client_hash_id, content_hash_id,
  SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN COALESCE(gsc_impressions,0) ELSE 0 END) AS impressions_14d,
  SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN COALESCE(gsc_clicks,0) ELSE 0 END) AS clicks_14d,
  SUM(CASE WHEN report_date <= DATE '2026-03-14' THEN COALESCE(gsc_sum_position,0) ELSE 0 END) AS sum_position_14d,
  COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-14' AND gsc_impressions > 0 THEN report_date END) AS active_days_14d,
  SUM(CASE WHEN report_date >= DATE '2026-03-15' THEN COALESCE(gsc_impressions,0) ELSE 0 END) AS impressions_late
FROM {base}
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

df = raw[raw["impressions_14d"] >= 100].copy()
df["ctr_14d"] = df["clicks_14d"] / df["impressions_14d"]
df["avg_position_14d"] = df["sum_position_14d"] / df["impressions_14d"]
df["declining"] = ((df["impressions_late"] / 17) < (df["impressions_14d"] / 14)).astype(int)

five = ["impressions_14d", "clicks_14d", "ctr_14d", "avg_position_14d", "active_days_14d"]
print("One row = one page. Pages:", len(df), "| share declining:", round(df["declining"].mean(), 3))
df[["client_hash_id", "content_hash_id"] + five + ["declining"]].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

One row = one page. Pages: 75526 | share declining: 0.511


,client_hash_id,content_hash_id,impressions_14d,clicks_14d,ctr_14d,avg_position_14d,active_days_14d,declining
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,211.0,1.0,0.004739,4.492891,14,0
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,237.0,0.0,0.000000,4.831224,14,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,385.0,1.0,0.002597,4.636364,14,1
5,client_62f4a7e64f5e0096,content_225dc9235023be5f,272.0,1.0,0.003676,11.029412,14,1
7,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,2484.0,0.0,0.000000,7.440016,14,1
8,client_62f4a7e64f5e0096,content_cfad137c1b04251b,168.0,0.0,0.000000,5.886905,14,0
9,client_62f4a7e64f5e0096,content_9e7c70abfbae371e,1630.0,0.0,0.000000,5.652761,14,1
10,client_62f4a7e64f5e0096,content_c0fc3b40ce00a5d7,165.0,1.0,0.006061,1.496970,14,1
11,client_62f4a7e64f5e0096,content_d79c0df3e1b437fb,161.0,0.0,0.000000,1.552795,14,1
12,client_62f4a7e64f5e0096,content_f414582a500a5cc0,800.0,5.0,0.006250,3.113750,14,0


Five features, all built only from March 1-14 (known on March 14):
- impressions_14d: total times the page was shown, March 1-14.
- clicks_14d: total clicks, March 1-14.
- ctr_14d: clicks divided by impressions, March 1-14 only.
- avg_position_14d: average search position, March 1-14 only.
- active_days_14d: number of days the page was shown, March 1-14.

Only pages with at least 100 impressions in March 1-14 are included.
The label "declining" uses March 15-31, so it is the future and is
never used as a feature.

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    X, y, g = df[cols], df["declining"], df["client_hash_id"]
    tr, te = next(GroupShuffleSplit(test_size=0.3, random_state=42).split(X, y, g))
    m = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X.iloc[tr], y.iloc[tr])
    return roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1])

print("Honest score (AUC):", round(quick_score(five), 3))

# The leak: this ratio is exactly how the label was made
df["leak_ratio"] = (df["impressions_late"] / 17) / (df["impressions_14d"] / 14)
print("With the leak (AUC):", round(quick_score(five + ["leak_ratio"]), 3))

df = df.drop(columns=["leak_ratio"])
print("After deleting the leak (AUC):", round(quick_score(five), 3))

Honest score (AUC): 0.586
With the leak (AUC): 1.0
After deleting the leak (AUC): 0.586


My honest score is 0.586 (AUC), which is only a little better than
guessing (0.5), so these five features carry a weak signal. When I
added a column built from the outcome (the late-versus-early
impressions ratio, which is exactly how the label is made), the score
jumped to [LEAK NUMBER], almost perfect. That is leakage: the column is
the answer in disguise, so it teaches me nothing. I deleted it and kept
the honest 0.586.

## 4. Data limits

Limitation: I use only one month (March 2026) from a dataset where
different clients have different amounts of history. My "declining"
label compares two halves of the same month, so normal ups and downs
or seasonality could look like a decline. It is a proxy, not proof
that a page lost visibility for a lasting reason.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.